In [1]:
# ==============================================
# 1. Imports
# ==============================================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna   # find best parameter
import mlflow   # track experiments
import mlflow.xgboost

/Users/apple/Documents/Projects/Nasa RUL MLE/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_df = pd.read_csv("/Users/apple/Documents/Projects/Nasa RUL MLE/data/processed/feature_engineered_train.csv")
test_df = pd.read_csv("/Users/apple/Documents/Projects/Nasa RUL MLE/data/processed/feature_engineered_test.csv")
rul_df = pd.read_csv("/Users/apple/Documents/Projects/Nasa RUL MLE/data/processed/clean_rul.csv")

In [3]:
# Features & target
X = train_df.drop(columns=["engine_id", "rul"])
y = train_df["rul"]

test_last = test_df.groupby("engine_id").last().reset_index()
X_test = test_last.drop(columns=["engine_id"])
y_test = rul_df["rul"]

In [4]:
# Train / Evaluation split
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
# Setup MLFlow
# Force MLflow to always use the root project mlruns folder
mlflow.set_tracking_uri("/Users/apple/Documents/Projects/Nasa RUL MLE/mlruns")
mlflow.set_experiment("NASA_Turbofan_RUL")

# Define optuna objective function with mlflow
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)

        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mae = float(mean_absolute_error(y_val, y_pred))
        r2 = float(r2_score(y_val, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [10]:
# Run optuna
study = optuna.create_study(direction="minimize")

with mlflow.start_run(run_name="xgboost_optuna"):
    study.optimize(objective, n_trials=30)

    # Log best result
    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_rmse", study.best_value)

[I 2026-04-09 13:14:26,094] A new study created in memory with name: no-name-ce3ef4c8-f137-47a4-a0a8-ea756387f47c
[I 2026-04-09 13:14:26,618] Trial 0 finished with value: 14.553217778967168 and parameters: {'n_estimators': 345, 'max_depth': 4, 'learning_rate': 0.11280943234338259, 'subsample': 0.8230570856305465, 'colsample_bytree': 0.587749893068584, 'min_child_weight': 2, 'gamma': 2.3491825383193476, 'reg_alpha': 0.0006183346879517927, 'reg_lambda': 0.00749375176917282}. Best is trial 0 with value: 14.553217778967168.
[I 2026-04-09 13:14:32,307] Trial 1 finished with value: 13.068204152482839 and parameters: {'n_estimators': 945, 'max_depth': 9, 'learning_rate': 0.040775160150266776, 'subsample': 0.7694424672137093, 'colsample_bytree': 0.8035257130974303, 'min_child_weight': 7, 'gamma': 1.5067306666021607, 'reg_alpha': 0.30520253382483215, 'reg_lambda': 0.006901250107213733}. Best is trial 1 with value: 13.068204152482839.
[I 2026-04-09 13:14:33,802] Trial 2 finished with value: 14.0

In [11]:
print("Best params:", study.best_params)
print("Best RMSE:", study.best_value)

Best params: {'n_estimators': 783, 'max_depth': 10, 'learning_rate': 0.015120088225847182, 'subsample': 0.9139945329715133, 'colsample_bytree': 0.9872223529763808, 'min_child_weight': 4, 'gamma': 3.5312523185894875, 'reg_alpha': 2.1668275605684396e-05, 'reg_lambda': 2.784849203207405e-08}
Best RMSE: 12.444927877015159


In [12]:
# ==============================================
# Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

y_pred_test = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2 = r2_score(y_test, y_pred_test)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.xgboost.log_model(best_model, name="model")

Final tuned model performance:
MAE: 14.586355209350586
RMSE: 20.26353397919948
R²: 0.762222409248352
